In [16]:
from pathlib import Path
from rich import print
import json
import polars as pl


In [23]:
path = Path("./q3-output")
assert path.exists(), "Output directory does not exist."

models = [
    "o3",
    "gemini_gemini-2.5-pro",
    "claude-opus-4-20250514",
    "claude-sonnet-4-20250514",
]

In [24]:
json_data = []
for model in models:
    model_path = path / model
    if not model_path.exists():
        print(f"[red]Model directory {model} does not exist.[/red]")
        continue

    files = list(model_path.glob("*.json"))
    files.sort()
    for file in files:
        text = file.read_text(encoding='utf-8')
        try:
            json_data.append(json.loads(text))
        except json.JSONDecodeError as e:
            print(f"[yellow]Error decoding JSON from {file}: {e}[/yellow]")
            continue

In [25]:
data = []
for d in json_data:
    comb = d['template_params']
    response = d['response']['choices'][0]['message']['content']
    response = response.replace("\n", "").replace("  ", "")
    combined = {**comb, **{'response': response}}
    data.append(combined)

pl.DataFrame(data).write_csv(path / "combined.csv")
